# AGCRS — Global Agricultural Yield, Climate Shocks & Parametric Insurance Risk

Reference reproducible study notebook auditing the FAO/World Bank 101-country crop yield panel (`1990–2013`), evaluating strict out-of-time (`2011–2013`) and Leave-One-Region-Out (LORO) models, distribution-free Split-Conformal 90% prediction intervals, non-linear climate shock econometrics (`+1°C`, `+2°C`, `-20%` drought), and parametric drought/heatwave crop insurance pricing with basis risk decomposition.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_PATH = PROJECT_ROOT / "artifacts" / "results.json"
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
print("Project:", results["project"])
print("Best Model (selected on 2009-2010 calibration RMSE):", results["best_model_name"])

## 1. Panel Integrity & Multi-Station Deduplication Audit

In [ ]:
audit = results["panel_audit"]
pd.DataFrame([
    {"Metric": "Raw CSV Rows", "Value": audit["raw_row_count"]},
    {"Metric": "Unique (Area, Item, Year) Panel Rows", "Value": audit["unique_panel_rows"]},
    {"Metric": "Redundant Multi-Station Rows Removed", "Value": audit["redundant_rows_removed"]},
    {"Metric": "Duplicate Inflation (%)", "Value": audit["duplicate_inflation_pct"]},
    {"Metric": "Multi-Station Temp-Varying Groups", "Value": audit["multi_station_temp_groups"]},
    {"Metric": "Yield-Varying Duplicate Groups", "Value": audit["yield_varying_dup_groups"]},
    {"Metric": "Missing Years", "Value": str(audit["missing_years"])},
    {"Metric": "Rainfall-Varying Countries", "Value": f"{audit['rainfall_varying_countries']} / {audit['n_countries']}"},
])

## 2. Out-of-Time (2011–2013) & Leave-One-Region-Out (LORO) Forecasting Benchmarks

In [ ]:
bench_rows = [
    {
        "Model": b["display_label"],
        "Cal RMSE": round(b["calibration_metrics"]["rmse"], 1),
        "Test RMSE": round(b["test_metrics"]["rmse"], 1),
        "Test MAE": round(b["test_metrics"]["mae"], 1),
        "Test MAPE (%)": round(b["test_metrics"]["mape_pct"], 2),
        "Test R2": round(b["test_metrics"]["r2"], 4),
        "LORO Mean R2": b["loro_mean_r2"],
        "LORO Mean RMSE": b["loro_mean_rmse"],
    }
    for b in results["model_benchmarks"]
]
display(pd.DataFrame(bench_rows))
display(Image(filename=str(PROJECT_ROOT / "images" / "agcrs_temporal_benchmark_conformal.png")))

## 3. Split-Conformal 90% Prediction Intervals (Standard vs Adaptive)

In [ ]:
conf = results["conformal_prediction"]
pd.DataFrame([
    {
        "Method": name,
        "Nominal": vals["nominal_coverage"],
        "Empirical": vals["empirical_coverage"],
        "Median Width (hg/ha)": vals["median_width_hg_ha"],
        "Mean Width (hg/ha)": vals["mean_width_hg_ha"],
        "Rel. Width (%)": vals["relative_width_pct"],
    }
    for name, vals in conf.items()
])

## 4. Non-Linear Climate Sensitivity Econometrics (+1°C, +2°C & -20% Drought)

In [ ]:
econ_cols = [
    "crop", "is_african_tropical_staple", "mean_temp_c",
    "shock_plus_1c_pct", "shock_plus_2c_pct", "shock_drought_minus_20pct", "fe_within_plus_2c_pct"
]
display(pd.DataFrame(results["climate_econometrics"]["crop_impacts"])[econ_cols].round(2))
display(Image(filename=str(PROJECT_ROOT / "images" / "agcrs_climate_shock_stress_test.png")))

## 5. TreeSHAP Yield Drivers & Parametric Crop Insurance Risk (<80% Baseline Strike)

In [ ]:
ins_cols = [
    "segment_name", "n_policies", "actual_shortfall_rate",
    "pure_premium_rate_pct", "parametric_loss_ratio", "fn_basis_risk_rate", "fp_basis_risk_rate"
]
display(pd.DataFrame(results["parametric_insurance"]["by_region"])[ins_cols].round(4))
display(Image(filename=str(PROJECT_ROOT / "images" / "agcrs_shap_yield_drivers.png")))
display(Image(filename=str(PROJECT_ROOT / "images" / "agcrs_parametric_insurance_risk.png")))